# Evaluation: reasoning on vs off (VLM parsing)

## Configuration and setup

Declaring the paths, models and lectures and loading the grounded VLM prompt

In [ ]:
import json, time, base64, os, re, unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

from system_prompt import get_system_prompt

ROOT     = Path.cwd().parent
PARSED   = ROOT / "data/eval/test_jsonfiles/reasoning"
REF      = ROOT / "data/reference_slides/machine_learning"
GOLDEN   = ROOT / "data/eval/test_jsonfiles/golden"
EVAL_OUT = ROOT / "data/eval/reasoning_evaluation_onoff"
EVAL_OUT.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env", override=True)
client      = OpenAI(base_url=os.getenv("GATEWAY_URL", ""), api_key=os.getenv("BEARER_TOKEN", ""))
VL_MODEL    = os.getenv("VL_MODEL_GATEWAY", "")
JUDGE_MODEL = os.getenv("INFERENCE_MODEL_GATEWAY", "")
assert VL_MODEL and JUDGE_MODEL, "VL_MODEL_GATEWAY / INFERENCE_MODEL_GATEWAY nicht gesetzt"
print("VL-Modell   :", VL_MODEL)
print("Judge-Modell:", JUDGE_MODEL)

SYSTEM_PROMPT = get_system_prompt()

LECTURES = [
    {"lecture": "ML_5_svm",
     "images": REF / "ML_5_svm",
     "golden": GOLDEN / "ML_5_svm_golden_parse.json"},
    {"lecture": "ML_9_neuronale_netze",
     "images": REF / "ML_9_neuronale_netze",
     "golden": GOLDEN / "ML_9_neuronale_netze_golden_parse.json"},
]

SETTING_LABEL = {False: "aus", True: "an"}
MODALITIES = ["grafik", "formel", "code"]

MAX_PAGES = None

## Schema and parse functions including the measurements

The slide schema and parse_slide, which parses one slide and also records latency and token usage for the cost comparison

In [ ]:
class VlmSlideOutput(BaseModel):
    title: str = Field(default="")
    page_content: str = Field(default="")
    context: str | None = Field(default=None)

def encode_image(img_path):
    return base64.b64encode(Path(img_path).read_bytes()).decode("utf-8")

def parse_slide(img_path, enable_thinking, retries=3):
    b64 = encode_image(img_path)
    for approach in range(retries):
        t0 = time.perf_counter()
        resp = client.chat.completions.create(
            model=VL_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": [
                    {"type": "text", "text": "Beschreibe die folgende Vorlesungsfolie wie im Systemprompt gefordert"},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                ]},
            ],
            response_format={"type": "json_object"},
            extra_body={"chat_template_kwargs": {"enable_thinking": enable_thinking}},
            max_tokens=8192,
            temperature=0,
        )
        latenz = time.perf_counter() - t0
        try:
            out = VlmSlideOutput.model_validate_json(resp.choices[0].message.content)
            return out, latenz, resp.usage
        except Exception as e:
            print(f"    Try {approach + 1} failed: {e}")
    raise RuntimeError(f"Parsing failed: {img_path}")


## Parse run (reasoning on and off)

Running each lecture twice, reasoning off and reasoning on

In [ ]:
parsed_chunks = {}  
runs_rows = []

for lec in LECTURES:
    lecture = lec["lecture"]
    pages = sorted(Path(lec["images"]).glob("page_*.png"),
                   key=lambda p: int(p.stem.split("_")[1]))
    if MAX_PAGES:
        pages = pages[:MAX_PAGES]

    for enable_thinking in [False, True]:
        label = SETTING_LABEL[enable_thinking]
        print(f"\n=== {lecture} | Reasoning {label} | {len(pages)} Folien ===")
        chunks = []
        for img_path in pages:
            page_no = int(img_path.stem.split("_")[1])
            out, latency, usage = parse_slide(img_path, enable_thinking)

            chunks.append({
                "id": f"{lecture}_page_{page_no}",
                "page_numbers": [page_no],
                "page_reference_path": str(img_path),
                "modul": "machine_learning",
                "lecture": lecture,
                "title": out.title,
                "page_content": out.page_content,
                "context": out.context or "",
            })
            runs_rows.append({
                "Lecture": lecture, "Reasoning": label, "slide": page_no,
                "latency_s": latency,
                "completion_tokens": getattr(usage, "completion_tokens", None),
                "total_tokens": getattr(usage, "total_tokens", None),
            })
            print(f"  Folie {page_no:2d}: {latency:5.1f}s, {getattr(usage, 'total_tokens', '?')} Tokens")

        parsed_chunks[(lecture, label)] = chunks
        out_path = PARSED / lecture / f"{lecture}_chunks_reasoning_{label}.json"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"  gespeichert: {out_path}")

runs_df = pd.DataFrame(runs_rows)

EVAL_OUT.mkdir(parents=True, exist_ok=True)
runs_path = EVAL_OUT / "reasoning_runs.csv"
runs_df.to_csv(runs_path, index=False, encoding="utf-8")
print("Messwerte gespeichert:", runs_path, "| Zeilen =", len(runs_df))

runs_df.head()

## Normalisation

Defining normalize (LaTeX to Unicode, case and punctuation folding) for formatting invariant recall

In [ ]:
LATEX = {
    r"\alpha": "α", r"\beta": "β", r"\gamma": "γ", r"\delta": "δ",
    r"\epsilon": "ε", r"\varepsilon": "ε", r"\zeta": "ζ", r"\eta": "η",
    r"\theta": "θ", r"\kappa": "κ", r"\lambda": "λ", r"\mu": "μ",
    r"\nu": "ν", r"\xi": "ξ", r"\pi": "π", r"\rho": "ρ", r"\sigma": "σ",
    r"\tau": "τ", r"\phi": "φ", r"\chi": "χ", r"\psi": "ψ", r"\omega": "ω",
    r"\leq": "≤", r"\le": "≤", r"\geq": "≥", r"\ge": "≥",
    r"\neq": "≠", r"\ne": "≠", r"\approx": "≈", r"\times": "×",
    r"\pm": "±", r"\infty": "∞", r"\sum": "∑", r"\partial": "∂",
    r"\nabla": "∇", r"\in": "∈", r"\rightarrow": "→", r"\to": "→",
}

def normalize(t: str) -> str:
    t = unicodedata.normalize("NFC", t)
    t = t.lower()
    for cmd in sorted(LATEX, key=len, reverse=True):    
        t = re.sub(re.escape(cmd) + r"(?![a-z])", LATEX[cmd], t)
    t = t.replace(",,", "").replace("``", "").replace("''", "")  
    t = re.sub(r'[„“”‚‘’»«"]', "", t)                    
    t = re.sub(r"\$+", " ", t)                           
    t = re.sub(r"[*#`>~]", " ", t)                   
    t = re.sub(r"…", "...", t)                          
    t = re.sub(r"[‐-―−]", "-", t)         
    t = re.sub(r"[•·‣▪]", " ", t)                       
    t = re.sub(r"(?m)^\s*[-→]\s+", " ", t)               
    t = re.sub(r";", " ", t)                            
    t = re.sub(r"\s*([.,:!?=≠≥≤≈×±])\s*", r"\1", t)     
    t = re.sub(r"\s+", " ", t)                        
    return t.strip()

## Combine nuggetinfos

Defining helpers to tell plain text nuggets from [GRAFIK]/[FORMEL]/[CODE] nuggets in the golden set, and to build the haystack a text nugget is searched in: the whole slide, so that both metrics answer the same question — is the information anywhere in the parse?

In [ ]:
BLOCK_PREFIXES = ("[GRAFIK]", "[FORMEL]", "[CODE]")

def is_text_nugget(n: str) -> bool:
    return not n.lstrip().startswith(BLOCK_PREFIXES)

def build_parsetext(chunk: dict) -> str:
    parts = []
    if chunk.get("title"):
        parts.append("Titel: " + chunk["title"])
    parts.append(chunk.get("page_content", "").replace("\\n", "\n"))
    return "\n".join(parts)

## Text recall (exact)

The recall function for the exact text recall metric

In [ ]:
def recall_counts(text_nuggets, chunk):
    if not text_nuggets:
        return 0, 0
    h = normalize(chunk)
    hits = sum(normalize(n) in h for n in text_nuggets)
    return hits, len(text_nuggets)

## LLM as a judge

Defining the LLM as judge and judge_item for the semantic block recall decision

In [ ]:
BLOCK_TYPES = ["grafik", "formel", "code"]

def build_fulltext(chunk):
    titel = "Titel: " + chunk["title"] + "\n" if chunk.get("title") else ""
    return titel + chunk.get("page_content", "")

cache = {}

def _call_judge(prompt):
    kwargs = dict(
        model=JUDGE_MODEL,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    try:
        return client.chat.completions.create(
            response_format={"type": "json_object"}, **kwargs
        )
    except Exception:
        return client.chat.completions.create(**kwargs)

def _extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n?", "", text)
        text = re.sub(r"\n?```$", "", text).strip()
    m = re.search(r"\{.*\}", text, re.DOTALL)   
    if m:
        text = m.group(0)
    return json.loads(text)

def judge_item(element, parse_text, retries=2):
    key = (element, parse_text)
    if key in cache:
        return cache[key]

    prompt = f"""
Du bist ein Evaluator fuer Informations-Recall in Folien-Parsing.

Aufgabe:
Pruefe ob, die Information des gegebenen Elements irgendwo im geparsten Text enthalten ist.

Wichtig:
- Es spielt KEINE Rolle, wo die Information steht (Text, Grafikbeschreibung, Formel, Code).
- Es spielt KEINE Rolle, wie sie formuliert ist.
- Entscheidend ist nur semantische Ähnlichkeit.
- Zusaetzlicher Inhalt im Parse ist irrelevant.
- Du bewertest nur Recall: enthalten oder nicht enthalten.

Element:
{element}

Geparste Folie:
{parse_text}

Antworte strikt als JSON:
{{"verdict": "covered" oder "missing", "reason": "kurze Begründung"}}
"""

    last_err = None
    for _ in range(retries):
        try:
            antwort = _call_judge(prompt)
            parsed = _extract_json(antwort.choices[0].message.content)
            verdict = str(parsed.get("verdict", "")).strip().lower()
            if verdict not in ("covered", "missing"):
                raise ValueError(f"unerwartetes verdict: {verdict!r}")
            ergebnis = {"verdict": verdict, "reason": parsed.get("reason", "")}
            cache[key] = ergebnis         
            return ergebnis
        except Exception as e:
            last_err = e

    return {"verdict": "invalid", "reason": f"Judge unbrauchbar nach {retries} Versuchen: {last_err}"}


In [ ]:
_golden = json.loads(Path(LECTURES[0]["golden"]).read_text(encoding="utf-8"))

probe = None
for slide in _golden:
    for block_type in BLOCK_TYPES:
        elements = slide.get(block_type, [])
        if elements:
            probe = elements[0]
            break
    if probe is not None:
        break

contains_text  = "Titel: Testfolie\n" + probe
unrelated_text = "Titel: Organisatorisches\nDie Klausur findet am 15. Maerz statt; bitte rechtzeitig anmelden."

print("ELEMENT:\n", probe, "\n")
print("contains it ->", judge_item(probe, contains_text))
print("unrelated   ->", judge_item(probe, unrelated_text))
print("\nExpected: 'covered' for the text that contains it, 'missing' for the unrelated one.")

## Recall helper (text + block)

Aggregating the two metric types over a parsed lecture

In [ ]:

def text_recall(golden, by_id):
    covered = total = 0
    for g in golden:
        if g["slide_id"] not in by_id:
            continue
        nuggets = [n for n in g["text"] if is_text_nugget(n)]
        hits, n = recall_counts(nuggets, build_parsetext(by_id[g["slide_id"]]))
        covered += hits
        total += n
    return covered, total

def block_recall(golden, by_id, modality):
    covered = total = 0
    for g in golden:
        if g["slide_id"] not in by_id:
            continue
        parse_text = build_fulltext(by_id[g["slide_id"]])
        for element in g.get(modality, []):
            result = judge_item(element, parse_text)
            verdict = result["verdict"]
            if verdict not in ("covered", "missing"):
                raise RuntimeError(
                    f"Judge lieferte kein gueltiges Urteil fuer {g['slide_id']} "
                    f"({modality}): {result['reason']}"
                )
            total += 1
            if verdict == "covered":
                covered += 1
    return covered, total


## Overall table

In [ ]:
golden_by_lecture = {lec["lecture"]: json.loads(Path(lec["golden"]).read_text(encoding="utf-8"))
                     for lec in LECTURES}

summary_rows = []
for enable_thinking in [False, True]:
    label = SETTING_LABEL[enable_thinking]
    runs  = runs_df[runs_df["Reasoning"] == label]

    covered = total = 0
    for lec in LECTURES:
        golden = golden_by_lecture[lec["lecture"]]
        by_id  = {c["id"]: c for c in parsed_chunks[(lec["lecture"], label)]}
        c, t = text_recall(golden, by_id)
        covered += c
        total   += t
        for modality in MODALITIES:
            c, t = block_recall(golden, by_id, modality)
            covered += c
            total   += t

    summary_rows.append({
        "Reasoning": label,
        "Folien (n)": int(len(runs)),
        "Ø Latenz/Folie (s)": runs["latency_s"].mean(),
        "Ø Tokens/Folie": runs["total_tokens"].mean(),
        "Gesamtzeit (s)": runs["latency_s"].sum(),
        "Gesamt-Tokens": runs["total_tokens"].sum(),
        "Gesamt-Recall": (covered / total) if total else None,
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(EVAL_OUT / "reasoning_summary.csv", index=False, encoding="utf-8")

for r in summary_rows:
    print(f"Reasoning {r['Reasoning']:3s} ({r['Folien (n)']} Folien): "
          f"Ø {r['Ø Latenz/Folie (s)']:5.1f} s/Folie, Ø {r['Ø Tokens/Folie']:6.0f} Tokens/Folie | "
          f"gesamt {r['Gesamtzeit (s)']:6.0f} s, {r['Gesamt-Tokens']:7.0f} Tokens | "
          f"Gesamt-Recall {r['Gesamt-Recall']:.1%}")
print("\ngespeichert:", EVAL_OUT / "reasoning_summary.csv")

(summary.style
        .hide(axis="index")
        .format({"Ø Latenz/Folie (s)": "{:.1f}",
                 "Ø Tokens/Folie": "{:.0f}",
                 "Gesamtzeit (s)": "{:.0f}",
                 "Gesamt-Tokens": "{:,.0f}",
                 "Gesamt-Recall":"{:.1%}"}, na_rep="–")
        .set_caption("Reasoning an vs. aus (beide Vorlesungen zusammengezogen)"))


In [ ]:
for lec in LECTURES:
    aus = parsed_chunks[(lec["lecture"], "aus")]
    an  = parsed_chunks[(lec["lecture"], "an")]
    identical = sum(a["page_content"] == b["page_content"] for a, b in zip(aus, an))
    print(f"{lec['lecture']:22s}: {len(aus)} Folien | identischer page_content in beiden Settings: {identical}")

split_rows = []
for label in ["aus", "an"]:
    for lec in LECTURES:
        golden = golden_by_lecture[lec["lecture"]]
        by_id  = {c["id"]: c for c in parsed_chunks[(lec["lecture"], label)]}

        covered, total = text_recall(golden, by_id)
        split_rows.append({"Reasoning": label, "Metrik": "text (exakt)", "covered": covered, "total": total})

        for modality in MODALITIES:
            covered, total = block_recall(golden, by_id, modality)
            split_rows.append({"Reasoning": label, "Metrik": f"{modality} (judge)", "covered": covered, "total": total})

split = (pd.DataFrame(split_rows)
           .groupby(["Reasoning", "Metrik"], as_index=False)[["covered", "total"]].sum())
split["recall"] = split["covered"] / split["total"]

print()
print(split.pivot(index="Metrik", columns="Reasoning", values=["covered", "total", "recall"]).to_string())


## Visualisation: overall costs (latency + tokens per slide)

Plotting the pooled cost side, average latency and tokens per slide, reasoning off vs on across both lectures, and saving the figure

In [ ]:
import matplotlib.pyplot as plt

settings = ["aus", "an"]
farben = {"aus": "#4C72B0", "an": "#DD8452"}

by_setting = summary.set_index("Reasoning")
n_folien = int(by_setting["Folien (n)"].iloc[0]) 

x = np.array([0, 0.5])          
breite = 0.32
labels = [f"Reasoning {s}" for s in settings]
balken = [farben[s] for s in settings]

def fmt_dauer(sekunden):
    m, s = divmod(int(round(sekunden)), 60)
    return f"{m}:{s:02d} min"

panels = [
    ("Ø Latenz/Folie (s)", "Ø Latenz pro Folie", "Sekunden", 1, lambda v: f"{v:.1f} s"),
    ("Ø Tokens/Folie", "Ø Tokens pro Folie", "Tokens",   1, lambda v: f"{v:.0f}"),
    ("Gesamtzeit (s)", "Gesamtzeit", "Sekunden", 1, fmt_dauer),
    ("Gesamt-Tokens", "Gesamt-Tokens", "Tokens",   1, lambda v: f"{v:,.0f}"),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 5))

for ax, (spalte, titel, ylabel, scale, labelfn) in zip(axes.flat, panels):
    roh   = [by_setting.loc[s, spalte] for s in settings]
    hoehe = [w * scale for w in roh]
    bars  = ax.bar(x, hoehe, breite, color=balken, zorder=3)
    ax.bar_label(bars, labels=[labelfn(w) for w in roh],
                 fontsize=8, fontweight="bold", padding=2)
    ax.set_title(titel, fontsize=9.5, fontweight="bold", pad=5)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_xlim(x[0] - 0.45, x[-1] + 0.45) 
    ax.tick_params(axis="y", labelsize=9)
    ax.margins(y=0.18)
    ax.grid(axis="y", alpha=0.25, zorder=0)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Reasoning an/aus – Kosten  (n = {n_folien} Folien je Einstellung)",
             fontsize=10.5, fontweight="semibold")
fig.tight_layout()
fig.savefig(EVAL_OUT / "reasoning_kosten.png", dpi=200, bbox_inches="tight")
print("gespeichert:", EVAL_OUT / "reasoning_kosten.png")
plt.show()
